## Preparación de Datos para K-Means Clustering
 
Se realiza un análisis de datos de clientes y prepara la información necesaria para aplicar un modelo de clustering K-Means. A continuación, se describen los pasos realizados:

 1. **Carga de Datos**: Se importan los datos de clientes desde una fuente pública.
 2. **Procesamiento de Fechas**: Se convierten las columnas de fechas al formato `datetime`.
 3. **Transformaciones y Cálculos**: Se realizan varias transformaciones y cálculos, como la conversión de valores monetarios, el cálculo del tiempo de vida del cliente, la frecuencia de compra, y la probabilidad de churn.
 4. **Preparación de Datos**: Se seleccionan las columnas relevantes y se verifican valores infinitos o nulos.
 5. **Exportación de Datos**: Los datos preparados se exportan a un archivo CSV para su uso en el modelo de clustering K-Means.

## **Importar librerias**
- Librerias necesarias para el funcionamiento
- importacion del sistema e inicializar el proyecto

In [ ]:
%pip install pymysql flask flask_migrate flask_bcrypt flask_jwt_extended flask_cors flask_sqlalchemy

  Using cached PyMySQL-1.1.1-py3-none-any.whl.metadata (4.4 kB)
Using cached PyMySQL-1.1.1-py3-none-any.whl (44 kB)
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 25.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [17]:
import sys
import os
import pandas as pd
import joblib
import numpy as np
# Asegurarce de que sea la ruta del proyecto
project_root = os.path.abspath('c:/Users/Sr. Sebastian/Documents/GitHub/CRMIA')
sys.path.append(project_root)

from app import create_app
from datetime import datetime
from app.models.customer import Customer
from app.models.segmento import Segmento
from app.models.catalogo_recomendaciones import CatalogoRecomendaciones
from app.models.recomendaciones import Recomendaciones
from sklearn.preprocessing import StandardScaler

## **Preparacion de datos**
Declaramos los datos para usarlos como caracteristicas
A continuación, se procederá a extraer los datos de clientes desde la base de datos, convertirlos en un DataFrame de pandas y realizar las transformaciones necesarias para preparar los datos para el modelo de clustering K-Means.

In [18]:
# Crear la aplicación Flask
app = create_app()

# Ejecutar dentro del contexto de la aplicación
with app.app_context():
    # Extraer datos de clientes desde la base de datos
    clientes = Customer.query.with_entities(
        Customer.id,
        Customer.total_compras,
        Customer.valor_medio_orden,
        Customer.probabilidad_churn,
        Customer.valor_cliente,
        Customer.fecha_creacion,
        Customer.ultima_compra
    ).all()

    # Convertir los datos a un DataFrame
    clientes_df = pd.DataFrame(clientes, columns=[
        'id',
        'total_compras',
        'valor_medio_orden',
        'probabilidad_churn',
        'valor_cliente',
        'fecha_creacion',
        'ultima_compra'
    ])
    print(clientes_df.columns)


Index(['id', 'total_compras', 'valor_medio_orden', 'probabilidad_churn',
       'valor_cliente', 'fecha_creacion', 'ultima_compra'],
      dtype='object')


## **Seleccion de caracteristicas**
Calculamos y Seleccionamos las caracteristicas requeridas por el modelo de entrenamiento. Se manejan las fechas, se calculan nuevas características como:
- la frecuencia de compra
- los días desde la última compra
- se manejan valores NaN e infinitos. 

Finalmente, se seleccionan las columnas requeridas para el modelo de entrenamiento.

In [19]:
# Manejo de fechas
clientes_df['fecha_creacion'] = pd.to_datetime(clientes_df['fecha_creacion'], errors='coerce')
clientes_df['ultima_compra'] = pd.to_datetime(clientes_df['ultima_compra'], errors='coerce')

# Calcular 'dias_ultima_compra' (manejo de clientes sin compras)
clientes_df['dias_ultima_compra'] = (datetime.now() - clientes_df['ultima_compra']).dt.days.fillna(9999)

# Calcular días como cliente
clientes_df['dias_como_cliente'] = (datetime.now() - clientes_df['fecha_creacion']).dt.days.replace(0, 1)

# Cálculo de frecuencia de compra
clientes_df['frecuencia_compra'] = (clientes_df['total_compras'] / (clientes_df['dias_como_cliente'] / 30)).round(2)

# **Manejo de valores NaN e infinitos**
clientes_df.fillna(0, inplace=True)
clientes_df.replace([np.inf, -np.inf], 0, inplace=True)

# Seleccionar columnas requeridas por el modelo de entrenamiento
prepared_data = clientes_df[[
    'id',
    'total_compras',
    'frecuencia_compra',
    'dias_ultima_compra',
    'valor_medio_orden',
    'probabilidad_churn',
    'valor_cliente'
]]

print(prepared_data.head())

   id  total_compras  frecuencia_compra  dias_ultima_compra  \
0   8            6.0               1.41                35.0   
1   9            3.0               1.32                32.0   
2  10            7.0               1.52                35.0   
3  11            5.0               1.55                35.0   
4  12            3.0               1.18                25.0   

   valor_medio_orden  probabilidad_churn  valor_cliente  
0            53000.0                0.10        2789.47  
1            37200.0                0.08        2066.67  
2            38500.0                0.13        2173.39  
3            33500.0                0.08        2018.07  
4            34667.0                0.00        1677.44  


C:\TEMP\ipykernel_13876\2799661230.py:15: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '0' has dtype incompatible with datetime64[ns], please explicitly cast to a compatible dtype first.
  clientes_df.fillna(0, inplace=True)


## **Resumen de pasos siguientes**
A continuación, se realizarán las siguientes acciones:

1. **Verificación de Datos**: Se comprobará si el conjunto de datos contiene valores inválidos (NaN o infinitos).
2. **Respuesta**: Se espera una `False` como respuesta
3. **Caso contrario**: si la respuesta no es `False` ejecutar del codigo de celda siguiente

In [20]:
# Verificar si el conjunto de datos tiene problemas generales
print("\n¿El conjunto de datos contiene algún valor inválido?")
print(prepared_data.isnull().values.any() or np.isinf(prepared_data).values.any())


¿El conjunto de datos contiene algún valor inválido?
False


In [ ]:
# Solo ejecutar si hay valores inválidos
if prepared_data.isnull().values.any() or np.isinf(prepared_data).values.any():
    prepared_data.loc[:, :] = prepared_data.replace([np.inf, -np.inf], 0.5)

## **Guardar el CSV**

In [21]:
# **Verificar datos finales**
print("\nDatos preparados para entrenamiento:")
print(prepared_data.columns)

output_dir = os.path.join("Data")
os.makedirs(output_dir, exist_ok=True)

output_path = os.path.join(output_dir, "kmeans_m2_data.csv")
prepared_data.to_csv(output_path, index=False)
print(f"\nArchivo '{output_path}' guardado exitosamente.")


Datos preparados para entrenamiento:
Index(['id', 'total_compras', 'frecuencia_compra', 'dias_ultima_compra',
       'valor_medio_orden', 'probabilidad_churn', 'valor_cliente'],
      dtype='object')

Archivo 'Data\kmeans_m2_data.csv' guardado exitosamente.
